### Notebook to Analyze Interneurons and whether they repeat

In [1]:
%qtconsole

In [1]:
import pickle, numpy as np, os, sys, pandas as pd, json
from matplotlib import pyplot as plt
from matplotlib.ticker import MaxNLocator
from scipy.stats import binom_test
from scipy.stats import mannwhitneyu
from scipy.stats import sem 
import repetition_manuscript_defaults as MDef
import ratterdam_Defaults as Def
import utility_fx as util
import ratterdam_RepetitionCoreFx as RepCore
from matplotlib.backends.backend_pdf import PdfPages
import williamDefaults as wmDef

In [2]:
def read_clust2(clustfile):
    '''open a cl-mazeX.X file and read spike times and max widths into 2 lists'''
    with open(clustfile) as clust:
        raw_clust_data = clust.read().splitlines()
    spikes = []
    widths = []
    for line in raw_clust_data[13:]:
        spikes.append(float(line.split(',')[-1]))
        widths.append(float(line.split(',')[-4]))
    return spikes, widths

In [3]:
rat_list = ['R765','R765','R781','R781','R808','R808','R859','R859','R886','R886']
day_list = ['RFD5','DFD4','D3','D4','D6','D7','D1','D2','D1','D2']


In [5]:
all_rates = []
all_widths = []
all_corrs = []
for rat, day in zip(rat_list, day_list):
    df = f'E:\\Ratterdam\\{rat}\\{rat}_RatterdamOpen_{day}\\'
    clustList, quals = util.getClustList(df,True)
    for clust, qual in zip(clustList, quals):
        if type(qual) == int and qual >= 3:
            spikes, widths = read_clust2(df+clust)
            rate = len(spikes) / ((spikes[-1]-spikes[0])/1e6)
            w = np.mean(widths)
            corr = np.correlate(np.diff(spikes)/1e6,np.diff(spikes)/1e6,'full')
            corr = corr / np.max(corr)
            all_rates.append(rate)
            all_widths.append(w)
            all_corrs.append(np.mean(corr))

In [62]:
fig, _ax = plt.subplots(1,1)

ax = fig.axes[0]
ax.scatter(np.log10(all_rates), all_widths,s=75,c='grey',edgecolor='black')
ax.set_xlabel("Log Mean Rate", fontsize=24)
ax.set_ylabel("Mean Width", fontsize=24)
plt.vlines(0.6,4,18,color='k',linestyle='--',linewidth=2)



We will use a threhold of log rate = 0.6 to differentiate (putative) interneurons and pyramidal neurons. Waveform width does not seem as useful so only will use rate
WH 20240317

In [4]:
threshold = 0.6 # in log units
INs_list = []
for rat, day in zip(rat_list, day_list):
    df = f'E:\\Ratterdam\\{rat}\\{rat}_RatterdamOpen_{day}\\'
    clustList, quals = util.getClustList(df,True)
    for clust, qual in zip(clustList, quals):
        if type(qual) == int and qual >= 3:
            spikes, widths = read_clust2(df+clust)
            rate = len(spikes) / ((spikes[-1]-spikes[0])/1e6)
            if np.log10(rate) >= threshold:
                INs_list.append({'rat':rat,'day':day,'clust':clust})


In [7]:
import json
with open('20240628_INlist.json', 'w') as f:
    json.dump(INs_list, f)

In [5]:
cmap = util.makeCustomColormap()

In [4]:
savePath = 'E:\\Ratterdam\\repetition_manuscript\\Supplementary_Figures\\interneurons\\'

In [8]:
with open (savePath + "20240628_INlist.json") as f:
    INs_list = json.load(f)
    f.close()

In [9]:
for i in range(len(INs_list)):
    inunit = INs_list[i]
    unit = RepCore.loadRepeatingUnit(inunit['rat'], inunit['day'], inunit['clust'], smoothing=2, vthresh=Def.velocity_filter_thresh)
    inunit['repunit'] = unit
    INs_list[i] = inunit


e:\UserData\Documents\GitHub\ratterdam\RatterdamOpen_Project\RateMapClass_William_20190308.py:203: FutureWarning: indices argument is deprecated and will be removed in version 0.20. To avoid this warning, please do not use the indices argument. Please see peak_local_max documentation for more details.
  peaks = peak_local_max(rateMap_no_nan, indices=False) # indices -> returns boolean of local extrema
e:\UserData\Documents\GitHub\ratterdam\RatterdamOpen_Project\RateMapClass_William_20190308.py:259: FutureWarning: indices argument is deprecated and will be removed in version 0.20. To avoid this warning, please do not use the indices argument. Please see peak_local_max documentation for more details.
  peaks = peak_local_max(field, min_distance=minPeakDistanceAwayBins, exclude_border=False, indices=False) # peaks must be minPeakDistanceAwayBins # of bins away from another peak
c:\Users\whockei1\Anaconda3\lib\site-packages\skimage\morphology\_deprecated.py:5: skimage_deprecation: Function

In [22]:
fig, ax = plt.subplots(5,5, figsize=(12,8))
for i, inunit in enumerate(INs_list):
    fig.axes[i].imshow(inunit['repunit'].repUnit.rateMap2D, origin='lower', aspect='auto', interpolation='None', 
                        cmap=cmap, vmax=np.nanpercentile(inunit['repunit'].repUnit.rateMap2D, 99),
                    extent=[wmDef.xedges[0], wmDef.xedges[-1], wmDef.yedges[0], wmDef.yedges[-1]])
    fig.axes[i].axis('off')
    t=inunit['clust'].split("\\")
    t2=t[1].split(".")
    fig.axes[i].set_title(f"{inunit['rat']} {inunit['day']} {t[0]} cell {t2[1]}")

In [11]:
%qtconsole

In [36]:
# with PdfPages(savePath+'interneurons.pdf') as pdf:
#     fig, ax= plt.subplots(5,5,figsize=(10,8))
#     for i, inunit in enumerate(INs_list):
#         unit = RepCore.loadRepeatingUnit(inunit['rat'], inunit['day'], inunit['clust'], smoothing=2, vthresh=Def.velocity_filter_thresh)
#         fig.axes[i].imshow(unit.repUnit.rateMap2D, origin='lower', aspect='auto', interpolation='None', 
#                         cmap=cmap, vmax=np.nanpercentile(unit.repUnit.rateMap2D, 90),
#                     extent=[wmDef.xedges[0], wmDef.xedges[-1], wmDef.yedges[0], wmDef.yedges[-1]])
#         fig.axes[i].axis('off')
#         fig.axes[i].set_title(f"{rat} {day} {unit.name}")
#     pdf.savefig()
#     plt.close()
    

e:\UserData\Documents\GitHub\ratterdam\RatterdamOpen_Project\RateMapClass_William_20190308.py:203: FutureWarning: indices argument is deprecated and will be removed in version 0.20. To avoid this warning, please do not use the indices argument. Please see peak_local_max documentation for more details.
  peaks = peak_local_max(rateMap_no_nan, indices=False) # indices -> returns boolean of local extrema
e:\UserData\Documents\GitHub\ratterdam\RatterdamOpen_Project\RateMapClass_William_20190308.py:259: FutureWarning: indices argument is deprecated and will be removed in version 0.20. To avoid this warning, please do not use the indices argument. Please see peak_local_max documentation for more details.
  peaks = peak_local_max(field, min_distance=minPeakDistanceAwayBins, exclude_border=False, indices=False) # peaks must be minPeakDistanceAwayBins # of bins away from another peak
c:\Users\whockei1\Anaconda3\lib\site-packages\skimage\morphology\_deprecated.py:5: skimage_deprecation: Function